# WorkflowContext from raw data

Ce notebook initialise un `WorkflowContext` a partir d'un dossier de fichiers bruts, puis fournit quelques cellules utiles pour inspecter `runs`, `configurations`, `issues` et generer les fichiers `refs_sub` / `refs_norm`.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

os.chdir(ROOT)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Repository root: {ROOT}")
print(f"Python path includes: {SRC}")

Repository root: /home/achennev/python/scarlet
Python path includes: /home/achennev/python/scarlet/src


In [2]:
from scarlet.workflow.context import initialize_workflow_context_from_raw_directory

RAW_DIR = ROOT / "data" / "SANSLLB" / "raw"
OUTPUT_DIR = ROOT / "data" / "SANSLLB" / "out"
INSTRUMENT_NAME = "sansllb"

RAW_DIR, OUTPUT_DIR

(PosixPath('/home/achennev/python/scarlet/data/SANSLLB/raw'),
 PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out'))

In [3]:
w = initialize_workflow_context_from_raw_directory(
    RAW_DIR,
    instrument_name=INSTRUMENT_NAME,
    output_dir=OUTPUT_DIR,
    overwrite=True
)

print(f"runs: {len(w.runs)}")
print(f"configurations: {len(w.configurations)}")
print(f"issues: {len(w.issues)}")
print(f"artifacts: {len(w.artifacts)}")

runs: 62
configurations: 2
issues: 4
artifacts: 64


In [4]:
sorted(
    [
        {
            "run_key": run_key.short(),
            "entity": run_key.entity,
            "mode": run_key.mode,
            "sample_name": run_key.sample_name,
            "path": str(path),
        }
        for run_key, path in w.runs.items()
    ],
    key=lambda row: row["path"],
)[:20]

[{'run_key': 'config_1:empty_beam:transmission:empty_beam_att5_ws_beamstop',
  'entity': 'empty_beam',
  'mode': 'transmission',
  'sample_name': 'empty_beam_att5_ws_beamstop',
  'path': '/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002339.nxs'},
 {'run_key': 'config_1:empty_cell:transmission:empty_cell',
  'entity': 'empty_cell',
  'mode': 'transmission',
  'sample_name': 'empty_cell',
  'path': '/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002341.nxs'},
 {'run_key': 'config_1:sample:transmission:H2O',
  'entity': 'sample',
  'mode': 'transmission',
  'sample_name': 'H2O',
  'path': '/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002342.nxs'},
 {'run_key': 'config_1:sample:transmission:S1_P_PB_25_2mm',
  'entity': 'sample',
  'mode': 'transmission',
  'sample_name': 'S1_P_PB_25_2mm',
  'path': '/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002343.nxs'},
 {'run_key': 'config_1:sample:transmission:S2_P_PB_60_2mm',
  'entity': 'sam

In [5]:
{
    config_id: {
        "wavelength": cfg.wavelength,
        "sample_detector_distance": cfg.sample_detector_distance,
        "has_collimation": cfg.collimation is not None,
        "notes": cfg.notes,
    }
    for config_id, cfg in w.configurations.items()
}

{'config_1': {'wavelength': 6.000461656749349,
  'sample_detector_distance': 2.5000035990000002,
  'has_collimation': True,
  'notes': None},
 'config_2': {'wavelength': 5.998901477173986,
  'sample_detector_distance': 8.999996799,
  'has_collimation': True,
  'notes': None}}

In [6]:
[
    {
        "level": issue.level,
        "where": issue.where,
        "message": issue.message,
        "key": issue.key,
    }
    for issue in w.issues
]

[{'level': 'WARN',
  'where': 'initialize_workflow_context_from_raw_directory',
  'message': 'Skipping non-HDF5 input file',
  'key': '/home/achennev/python/scarlet/data/SANSLLB/raw/mask_GQ.edf'},
 {'level': 'WARN',
  'where': 'initialize_workflow_context_from_raw_directory',
  'message': 'Skipping non-HDF5 input file',
  'key': '/home/achennev/python/scarlet/data/SANSLLB/raw/mask_PQ.edf'},
 {'level': 'WARN',
  'where': 'initialize_workflow_context_from_raw_directory',
  'message': 'Duplicate run key detected; overwriting previous file',
  'key': 'config_1:dark:scattering:Cd'},
 {'level': 'WARN',
  'where': 'initialize_workflow_context_from_raw_directory',
  'message': 'Duplicate run key detected; overwriting previous file',
  'key': 'config_2:dark:scattering:Cd'}]

In [7]:
from scarlet.workflow.context import generate_reference_files_from_workflow_context

generate_reference_files_from_workflow_context(w)
print("refs_sub:", {k: str(v) for k, v in sorted(w.refs_sub_files.items())})
print("refs_norm:", {k: str(v) for k, v in sorted(w.refs_norm_files.items())})

refs_sub: {'config_1': '/home/achennev/python/scarlet/data/SANSLLB/out/refs_sub_config_1.nxs', 'config_2': '/home/achennev/python/scarlet/data/SANSLLB/out/refs_sub_config_2.nxs'}
refs_norm: {'config_1': '/home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_1.nxs', 'config_2': '/home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_2.nxs'}


In [ ]:
from scarlet.gui import run_mask_editor

# run_mask_editor()

In [8]:
from scarlet.workflow.context import write_runs_report_csv

In [9]:
file = write_runs_report_csv(w, overwrite=True)
print(f"Runs report written to: {file}")

Runs report written to: /home/achennev/python/scarlet/data/SANSLLB/out/runs_report.csv


In [10]:
# test change with respect to modified run csv file
# from scarlet.workflow.context import update_workflow_context_from_runs_report_csv
# update_workflow_context_from_runs_report_csv(w,"data/SANSLLB/out/runs_report_modified.csv")
# update_workflow_context_from_runs_report_csv(w,"data/SANSLLB/out/runs_report.csv")
# file = write_runs_report_csv(w, csv_path="data/SANSLLB/out/runs_report_updated.csv", overwrite=True)

In [11]:
# I want to print the updated runs from the workflow context after the update
print("Updated runs:")
for run_key, path in w.runs.items():
    print(f"{run_key.short()}: {path}")

Updated runs:
config_1:empty_beam:transmission:empty_beam_att5_ws_beamstop: /home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002339.nxs
config_1:dark:scattering:Cd: /home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002356.nxs
config_1:empty_cell:transmission:empty_cell: /home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002341.nxs
config_1:sample:transmission:H2O: /home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002342.nxs
config_1:sample:transmission:S1_P_PB_25_2mm: /home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002343.nxs
config_1:sample:transmission:S2_P_PB_60_2mm: /home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002344.nxs
config_1:sample:transmission:S3_P_PBK_40_2_2mm: /home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002345.nxs
config_1:sample:transmission:S4_P_BC_25_2mm: /home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002346.nxs
config_1:sample:transmission:S5_P_BC_60_2mm: /home/achennev/pytho

In [12]:
w.issues

[Issue(level='WARN', message='Skipping non-HDF5 input file', where='initialize_workflow_context_from_raw_directory', key='/home/achennev/python/scarlet/data/SANSLLB/raw/mask_GQ.edf', when_utc='2026-05-27T13:37:53.716458+00:00', meta={}),
 Issue(level='WARN', message='Skipping non-HDF5 input file', where='initialize_workflow_context_from_raw_directory', key='/home/achennev/python/scarlet/data/SANSLLB/raw/mask_PQ.edf', when_utc='2026-05-27T13:37:53.716500+00:00', meta={}),
 Issue(level='WARN', message='Duplicate run key detected; overwriting previous file', where='initialize_workflow_context_from_raw_directory', key='config_1:dark:scattering:Cd', when_utc='2026-05-27T13:37:54.266120+00:00', meta={'previous_path': '/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002340.nxs', 'new_path': '/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002356.nxs'}),
 Issue(level='WARN', message='Duplicate run key detected; overwriting previous file', where='initialize_workflow_cont

In [13]:
w.refs_norm_files

{'config_1': PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_1.nxs'),
 'config_2': PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_2.nxs')}

In [18]:
from scarlet.workflow import update_reference_masks_from_workflow_context

w = update_reference_masks_from_workflow_context(w)

scarlet.workflow.context.WorkflowContext